# Каталоги скоплений

Исходный учебный материал по данным. Загрузки направлены в `data/raw/`; обработанные изображения находятся в `data/processed/`. В отличие от итогового анализа этот ноутбук требует внешних данных и ручного выбора примеров. [Воспроизведение](../../docs/reproduction.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

**Скопления галактик** - наиболее массивные гравитационно-связанные объекты во Вселенной

Вам предстоит работать в картах ACT+Planck и в этом обзоре были созданы крупнейшие каталоги скоплений галактик, найденные с помощью эффекта [Сюняеыва-Зельдовича](https://ru.wikipedia.org/wiki/Эффект_Сюняева_—_Зельдовича)

Масса скопления определяются относительно двух характеристик плотности: средней плотности вещества во Вселенной (m) или критической плотности Вселенной (crit или c, $\rho_{crit} = \frac{3H^2}{8\pi G}$). Для каждого из этих эталонов вводится значение плотностного контраста $\Delta$, указывающее, во сколько раз плотность в пределах заданного радиуса превышает выбранную опорную величину. Для численных симуляций наиболее часто используемым является $\Delta = 200$, который наиболее близок к вириальному радиусу скопления. В микроволновых и рентгеновских исследованиях, напротив, предпочтение обычно отдаётся контрасту $\Delta = 500$, поскольку соответствующий радиус охватывает область, где предположение о гидростатическом равновесии наиболее применимо. 

**Если будете брать массу, то индекс должен быть 500 и если есть m или c, то c**

Для оценки массы в СЗ используется интегральный Y-параметр, который получается путем интегрирования сигнала tSZ по объему скопления V:
$$Y = D_A^{-2}\frac{k_B \sigma_T}{m_e c^2} \int n_e T_e dV= D_A^{-2} \frac{\sigma_T}{m_e c^2} \int P_e dV$$
где $P_e$ - давление газа ($P_e = n_e k_B T_e$, $k_B$ - постоянная Больцмана, $T_e$ и $n_e$ - температура  и плотности числа электронов, $\sigma_T$ - сечение Томсона), $m_e c^2$ - масса покоя электрона, $D_A$ - угловое расстояние.

или максимальный у-параметр (как в коллаборации ACT):
$$y_0 = 10^{A_0} E(z)^2 \left(\frac{M_{500c}}{M_{pivot}}\right)^{1+B_0} Q(\theta_{500c})f_{rel}(M_{500c}, z)$$
где $E(z) = \sqrt{\Omega_m (1 + z)^3 + \Omega_\Lambda}$ - безразмерный параметр Хаббла, $10^{A_0}$ - нормировка, $B_0$, $M_{pivot}$ - константы, $Q(\theta_{500c})$ - функция несоотвествия фильтра, $f_{rel}$ - релятивисткие поправки (см. [статью](https://arxiv.org/abs/2009.11043))

А сам y/Y параметр получается из микроволновых карт на разных частотах с помощью многочастотной фильтрации и показывает, насколько скопление исказило чернотельный спектр реликтового фона. Это искажение меняет знак на частоте 217 ГГц: ослабляет излучение ниже этой частоты и усиливает выше.

In [ ]:
(PROJECT_ROOT / "data/raw").mkdir(parents=True, exist_ok=True)

**Важно, брать только значения масс с известными красными смещениями (z, redshift)** (z > 0)

In [ ]:
%pip install astroquery

In [ ]:
from galaxy_mass_estimation import catalogs as load
import pandas as pd, numpy as np
import os
import matplotlib.pyplot as plt

# Основной каталог

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from galaxy_mass_estimation import catalogs as load

actdr6 = ["ACTDR6"]
output_folder = str(PROJECT_ROOT / "data/raw")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

catal = load.download_catalogs(actdr6, base_path=PROJECT_ROOT / "data/raw")

bins = np.arange(14, 15.3, 0.05)

plt.figure(figsize=(6, 4))

# ⬇️ Сохраняем результаты гистограмм
counts_corr, edges_corr, _ = plt.hist(
    np.log10(catal["ACTDR6"].M500c * 1e14),
    bins=bins,
    alpha=0.7,
    color="steelblue",
    edgecolor="none",
    linewidth=0.5,
    label="M500c",
)

counts_uncorr, edges_uncorr, _ = plt.hist(
    np.log10(catal["ACTDR6"].M500cUncorr * 1e14),
    bins=bins,
    alpha=0.7,
    color="red",
    histtype="step",
    linewidth=2,
    label="M500cUncorr",
)

plt.xlabel(r"log$_{10}$(M$_{500c}$ / M$_{\odot}$)", fontsize=12)
plt.ylabel("N", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# --- теперь у тебя есть массивы ---
print("bins:", bins)
print("counts_corr:", counts_corr)
print("counts_uncorr:", counts_uncorr)

In [ ]:
actdr6 = ["ACTDR6"]
output_folder = str(PROJECT_ROOT / "data/raw")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
catal = load.download_catalogs(actdr6, base_path=PROJECT_ROOT / "data/raw")

In [ ]:
bins = np.arange(14, 15.3, 0.05)
plt.figure(figsize=(6, 4))
plt.hist(
    np.log10(catal["ACTDR6"].M500c * 1e14),
    bins=bins,
    alpha=0.7,
    color="steelblue",
    edgecolor="none",
    linewidth=0.5,
)
plt.hist(
    np.log10(catal["ACTDR6"].M500cUncorr * 1e14),
    bins=bins,
    alpha=0.7,
    color="red",
    histtype="step",
    linewidth=2,
)
plt.xlabel(r"log$_{10}$(M$_{500c}$ / M$_{\odot}$)", fontsize=12)
plt.ylabel("N", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

$M_{500c}$ массы лучше описывают сигнал с карт. Т.е. если вы просимулируете скопление с такой массой и вычтете профиль из карты, то остатка будешь меньше чем в случае вычитания $M_{500cUncorr}$ масс. Но последние лучше согласуются с измерениями масс, полученными в других миссиях

In [ ]:
columns = [
    "name",
    "RADeg",
    "decDeg",
    "redshift",
    "redshiftType",
    "M500c",
    "M500cUncorr",
    "theta500Arcmin",
]
df = catal["ACTDR6"][columns]
df.to_csv(PROJECT_ROOT / "data/raw/clusters_selected.csv", index=False)

In [ ]:
df.head()

# Дополнительные каталоги

In [ ]:
dop = [
    "PSZ2",
    "PSZ1",
    "PSZ1_Barenna",
    "PSZ-MCMF",
    "ACTDR6",
    "ACTDR5",
    "PSZSPT",
    "ComPRASS",
    "ACT-MCMF",
    "SPT-SZ",
    "SPT2500D",
    "SPTECS",
    "SPTDEEP",
    "SPTFAR",
    "SPT100deg",
    "SPT-MCMF",
]